# Inquiry Confirmation

Four inquiries in one run: BiRefNet and the parser on the GPU, the call-2 upscale, the second crop, and garment-only references.

## 1 · Settings

In [ ]:
DRIVE_PROJECT_DIR = "Side projects and shi"
A100_CAD_PER_HOUR = 0.689
BFL_REPO, BFL_REV = "black-forest-labs/FLUX.2-klein-4B", "e7b7dc27f91deacad38e78976d1f2b499d76a294"
PR_REPO, PR_REV = "Photoroom/FLUX.2-klein-4b-fp8-diffusers", "408c457f3589e17a1be1dae5bf0dcaf09cd4985f"
PR_SUB = "transformer_bf16"
BRANCH = "https://github.com/101011101/magichour_takehome/raw/v3.3-lock"
MATRIX = "v39_set.csv"
ARMS = ("SCALE", "NOSCALE", "CROP2")
CROP_WORKERS = 4
PRODUCT = ["g001", "g002", "g008", "g010", "g016", "g017", "g021", "g023", "g026", "g028"]
PEOPLE = ["dualuse_emma_watson_black_blazer_armscrossed",
          "dualuse_hugh_jackman_grey_suit_outdoor",
          "dualuse_woman_top_denim_skirt_nonceleb"]
SEEDS = (46, 47)
CALL1_SEED = 46

## 2 · Environment

In [ ]:
import glob
import json
import os
import subprocess
import sys

PIP = [sys.executable, "-m", "pip"]
PROBE = "/content/ort_report.py"

ORT_REPORT = """
import ctypes, glob, json, os, site, sys
try:
    import onnxruntime as ort
except ImportError:
    print(json.dumps({"installed": False, "available": [], "loads": False, "error": "not installed"}))
    sys.exit(1)
info = {"installed": True, "version": ort.__version__, "device": ort.get_device(),
        "available": ort.get_available_providers(), "loads": False, "error": ""}
for d in sorted({d for p in site.getsitepackages() for d in glob.glob(os.path.join(p, "nvidia", "*", "lib"))}):
    for f in os.listdir(d):
        if ".so" in f and any(k in f for k in ("cudart", "cublas", "cudnn", "cufft", "curand")):
            try:
                ctypes.CDLL(os.path.join(d, f), mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass
so = glob.glob(os.path.dirname(ort.__file__) + "/capi/libonnxruntime_providers_cuda.so")
if not so:
    info["error"] = "wheel carries no CUDA provider library"
else:
    try:
        ctypes.CDLL(so[0], mode=ctypes.RTLD_GLOBAL)
        info["loads"] = True
    except OSError as e:
        info["error"] = str(e).strip().split("\\n")[-1]
print(json.dumps(info))
sys.exit(0 if info["loads"] else 1)
"""
open(PROBE, "w").write(ORT_REPORT)


def ort_report():
    r = subprocess.run([sys.executable, PROBE], capture_output=True, text=True)
    out = r.stdout.strip().splitlines()
    if out and out[-1].startswith("{"):
        return json.loads(out[-1])
    tail = (r.stderr.strip().splitlines() or ["probe produced no output"])[-1]
    return {"installed": True, "available": [], "loads": False, "error": tail[:120]}


import torch

before = ort_report()
print(f"{'gpu':28s} {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
print(f"{'torch CUDA':28s} {torch.version.cuda}")
print(f"{'onnxruntime':28s} {before.get('version', 'not installed')}")
print(f"{'registers CUDA provider':28s} {'CUDAExecutionProvider' in before['available']}")
print(f"{'CUDA provider actually loads':28s} {before['loads']}  {before['error']}")
assert "A100" in torch.cuda.get_device_name(0), "Runtime > Change runtime type > A100"

## 3 · Install

In [ ]:
subprocess.run(PIP + ["install", "-q", "-U", "diffusers", "transformers", "accelerate",
                      "sentencepiece", "protobuf", "huggingface_hub"], check=True, capture_output=True)
subprocess.run(PIP + ["install", "-q", "mediapipe==1.0.1"], check=True, capture_output=True)
subprocess.run(PIP + ["uninstall", "-q", "-y", "onnxruntime", "onnxruntime-gpu"], capture_output=True)

chosen = None
for spec in ["", "==1.22.0", "==1.21.1", "==1.20.1", "==1.19.2", "==1.18.1"]:
    label = spec.lstrip("=") or "newest"
    if subprocess.run(PIP + ["install", "-q", f"onnxruntime-gpu{spec}"], capture_output=True).returncode:
        print(f"onnxruntime-gpu {label:8s} no installable wheel")
        continue
    info = ort_report()
    if info["loads"]:
        chosen = info["version"]
        print(f"onnxruntime-gpu {info['version']:8s} CUDA provider loads")
        break
    print(f"onnxruntime-gpu {info.get('version', label):8s} registers "
          f"{'CUDAExecutionProvider' in info['available']}, loads False - {info['error']}")
if chosen is None:
    raise RuntimeError("no onnxruntime-gpu build loads CUDA on this runtime")

subprocess.run(PIP + ["uninstall", "-q", "-y", "opencv-python", "opencv-python-headless",
                      "opencv-contrib-python"], capture_output=True)
subprocess.run(PIP + ["install", "-q", "opencv-contrib-python-headless==5.0.0.93"],
               check=True, capture_output=True)

import cv2

print(f"{'opencv':16s} {cv2.__version__}")
print(f"{'guidedFilter':16s} {hasattr(cv2.ximgproc, 'guidedFilter')}")
assert hasattr(cv2.ximgproc, "guidedFilter"), "plain opencv shadowed opencv-contrib - Runtime > Restart"

## 4 · Downloads

In [ ]:
import csv
import shutil
import zipfile

from google.colab import drive

shutil.rmtree("/content/inquiry", ignore_errors=True)
subprocess.run(["wget", "-q", "-O", "/content/v39.zip", f"{BRANCH}/v39_bundle.zip"], check=True)
with zipfile.ZipFile("/content/v39.zip") as z:
    z.extractall("/content/inquiry")
os.chdir("/content/inquiry")
sys.path.insert(0, "lib")
for f in ("lib/klein_local.py", "lib/v3lib.py", "lib/run_v36.py", "lib/run_v39.py",
          "lib/ironman_bc_crop.py", "lib/garment_crop.py", "lib/phase3_variants.py", MATRIX):
    if not os.path.exists(f):
        raise FileNotFoundError(f"bundle incomplete: {f} - is v39_bundle.zip pushed to v3.3-lock?")

drive.mount("/content/drive")
MYDRIVE = "/content/drive/MyDrive"
BASE = os.path.join(MYDRIVE, DRIVE_PROJECT_DIR)
if not os.path.isdir(BASE):
    raise FileNotFoundError(f"Drive project dir not found: {BASE}")
cands = [os.path.join(MYDRIVE, "hf_cache"), os.path.join(BASE, "tryon_models", "hf_cache"),
         os.path.join(BASE, "hf_cache")]
os.environ["HF_HOME"] = next((c for c in cands if os.path.isdir(os.path.join(c, "hub"))), cands[0])
os.environ["V3_MODEL_DIR"] = os.path.join(BASE, "v3_models")
os.environ["V2_ORT_GPU"] = "1"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.makedirs(os.environ["V3_MODEL_DIR"], exist_ok=True)
os.makedirs("/content/v2/runs/.models", exist_ok=True)
for name in ("BiRefNet_lite.onnx", "pose_landmarker_lite.task", "selfie_multiclass_256x256.tflite"):
    src = os.path.join(os.environ["V3_MODEL_DIR"], name)
    dst = f"/content/v2/runs/.models/{name}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)

rows = list(csv.DictReader(open(MATRIX)))
garments = sorted({r["garment"] for r in rows})
stems = {r["person"] for r in rows} | set(garments) | set(PEOPLE)


def pick(pattern, exclude=None, required=True):
    zs = [z for z in sorted(glob.glob(os.path.join(BASE, "v3_runs", pattern)))
          if not (exclude and exclude in os.path.basename(z))]
    if not zs and required:
        raise FileNotFoundError(f"no {pattern} under Drive v3_runs/")
    return zs[-1] if zs else None


want = {pick("v34_ironman2_*.zip", exclude="_bc_"): {f"inputs/{s}.jpg" for s in stems},
        pick("v34_ironman2_bc_*.zip"):
            {f"refs/{g}__bald.jpg" for g in garments} | {f"refs/{g}__BC.jpg" for g in garments}}
for zp, names in want.items():
    with zipfile.ZipFile(zp) as z:
        missing = names - set(z.namelist())
        if missing:
            raise FileNotFoundError(f"{os.path.basename(zp)} is missing {sorted(missing)[:3]}")
        z.extractall("run", members=sorted(names))
    print(f"{os.path.basename(zp):44s} {len(names):3d} files")

os.makedirs("inq/inputs", exist_ok=True)
for g in PRODUCT:
    dst = f"inq/inputs/{g}.jpg"
    if not os.path.exists(dst):
        subprocess.run(["wget", "-q", "-O", dst, f"{BRANCH}/test_set1/garments/{g}.jpg"], check=True)
    if os.path.getsize(dst) < 1000:
        raise RuntimeError(f"{g}.jpg did not download")
print(f"{'product shots':44s} {len(PRODUCT):3d} files")

from huggingface_hub import snapshot_download

import klein_local as K

bfl = snapshot_download(BFL_REPO, revision=BFL_REV,
                        ignore_patterns=["flux-2-klein-4b.safetensors", "*.jpg"])
pr = snapshot_download(PR_REPO, revision=PR_REV, allow_patterns=[f"{PR_SUB}/*"])
K.load(repo=bfl, transformer=(pr, PR_SUB))
if not K.info().get("transformer", "").startswith(pr):
    raise RuntimeError("the transformer swap did not take")
print(f"{'klein':44s} {K.info()}")

edits_v39 = len(rows) * 2 + sum(1 for r in rows if r.get("noscale_noop") != "1")
calls = len(garments) * 2 + edits_v39 + len(PRODUCT) + len(PRODUCT) * len(PEOPLE) * len(SEEDS) * 2
print(f"\n{'estimated klein calls':44s} {calls}")
print(f"{'estimated GPU time':44s} ~{calls * 2.3 / 60:.0f} min")
print(f"{'estimated cost':44s} ~CAD {calls * 2.3 / 3600 * A100_CAD_PER_HOUR:.2f}")

## 5 · Inquiry 1 — BiRefNet and the SCHP parser on the GPU

In [ ]:
import time

import numpy as np

import garment_crop as GC
import phase3_variants as P
from ironman_bc_crop import crop_bc


def sessions(device):
    os.environ["V2_ORT_GPU"] = "1" if device == "gpu" else "0"
    GC._STATE.pop("biref", None)
    GC._STATE.pop("biref_prov", None)
    P._HP.clear()
    GC._biref()
    P._parser()
    if P._HP.get("m") is None:
        raise RuntimeError("the SCHP parser did not load at all")
    return GC._STATE["biref_prov"], P._HP["m"].get_providers()[0]


biref_prov, parser_prov = sessions("gpu")
for name, prov in (("BiRefNet", biref_prov), ("SCHP parser", parser_prov)):
    print(f"{name:16s} {prov}")
    if prov != "CUDAExecutionProvider":
        raise RuntimeError(f"{name} fell back to {prov} - the crops are not on the GPU")

probe = sorted(glob.glob("run/refs/*__bald.jpg"))[0]
img = cv2.imread(probe)


def timings(device):
    sessions(device)
    GC.biref_matte(img, f"warmup_{device}", True)
    P.parse_human(img)
    _, dt_biref = GC.biref_matte(img, f"timing_{device}", True)
    t = time.time()
    P.parse_human(img)
    return dt_biref, time.time() - t


gpu_biref, gpu_parser = timings("gpu")
cpu_biref, cpu_parser = timings("cpu")
print(f"\n{os.path.basename(probe)}  {img.shape[1]}x{img.shape[0]}\n")
print(f"{'stage':16s} {'GPU s':>8s} {'CPU s':>8s} {'speed-up':>9s}")
for name, g, c in (("BiRefNet", gpu_biref, cpu_biref), ("SCHP parser", gpu_parser, cpu_parser)):
    print(f"{name:16s} {g:8.2f} {c:8.2f} {c / g:8.1f}x")
print(f"\nVERDICT  both ONNX models run on CUDA; crops are {(cpu_biref + cpu_parser) / (gpu_biref + gpu_parser):.1f}x faster on the GPU")

## 6 · Inquiry 1 — do GPU crops reproduce the CPU references of record (T1)

In [ ]:
sessions("gpu")
shutil.rmtree(GC.CACHE_DIR, ignore_errors=True)

parity = []
for ref in sorted(glob.glob("run/refs/*__BC.jpg")):
    stem = os.path.basename(ref)[:-len("__BC.jpg")]
    bald = f"run/refs/{stem}__bald.jpg"
    if not os.path.exists(bald):
        continue
    a = cv2.imread(ref)
    b, cranium = crop_bc(cv2.imread(bald), f"gpu_{stem}")
    if abs(a.shape[0] - b.shape[0]) > 8 or abs(a.shape[1] - b.shape[1]) > 8:
        parity.append((stem, float("nan"), float("nan"), False,
                       f"shape {b.shape[1]}x{b.shape[0]} vs {a.shape[1]}x{a.shape[0]}"))
        continue
    d = np.abs(a.astype(np.float32) - cv2.resize(b, (a.shape[1], a.shape[0])).astype(np.float32))
    parity.append((stem, float(d.mean()), float(d.max()), float(d.mean()) <= 4.0, ""))

failed = [r for r in parity if not r[3]]
print(f"{'reference':54s} {'MAD':>7s} {'max':>5s}  note")
for stem, mad, mx, ok, note in parity:
    print(f"{stem[:54]:54s} {mad:7.2f} {mx:5.0f}  {note}")
gpu_parity = bool(parity) and not failed
print(f"\nVERDICT  {len(parity) - len(failed)}/{len(parity)} within MAD 4.0 - "
      f"{'GPU crops match the record, production can run them on the GPU' if gpu_parity else 'GPU crops DIVERGE, keep production crops on CPU'}")

## 7 · Inquiries 2 and 3 — the upscale, and the second crop

In [ ]:
sessions("cpu")
shutil.rmtree(GC.CACHE_DIR, ignore_errors=True)

import run_v39 as V

V.main(MATRIX, arms=ARMS, gpu_usd_per_hour=A100_CAD_PER_HOUR, crop_workers=CROP_WORKERS, limit=1)

In [ ]:
V.main(MATRIX, arms=ARMS, gpu_usd_per_hour=A100_CAD_PER_HOUR, crop_workers=CROP_WORKERS)
v39_meta = json.load(open("run/meta/v39_meta.json"))
v39_cost = json.load(open("run/meta/cost_v39.json"))
noop = sum(1 for c in v39_meta["cells"].values() if c.get("NOSCALE_noop"))
made = {a: len(glob.glob(f"run/gen/*__{a}__*.jpg")) for a in ARMS}
want = {"SCALE": len(rows), "NOSCALE": len(rows) - noop, "CROP2": len(rows)}
for a in ARMS:
    print(f"{a:8s} {made[a]}/{want[a]} cells")
short = [a for a in ARMS if made[a] < want[a]]
if short:
    raise RuntimeError(f"incomplete: {short} - rerun this cell, it resumes")
print(f"\nVERDICT  {sum(made.values())} outputs in {v39_cost['wall_minutes']:.0f} min; "
      f"{noop} NOSCALE no-ops; judge them on the marking page - the run cannot answer these two")

## 8 · Inquiry 4 — garment-only references

In [ ]:
import v3lib as L
import run_v36 as V36

JPG = [cv2.IMWRITE_JPEG_QUALITY, 95]
for sub in ("inq/refs", "inq/gen", "inq/meta", "inq/in1mp"):
    os.makedirs(sub, exist_ok=True)


def crop_nobald(img, stem):
    M = P.masks(img, stem, cranium=False)
    x0, y0, x1, y1 = GC.bbox_of((M["subject"] > 0.5).astype(np.uint8), img.shape[:2])
    return P.flatten(img[y0:y1, x0:x1], M["noface"][y0:y1, x0:x1], P.WHITE)


def white_frac(im):
    return float((im.min(axis=2) > 245).mean())


inq = {}
for g in PRODUCT:
    raw = V.normalise(cv2.imread(f"inq/inputs/{g}.jpg"))
    cv2.imwrite(f"inq/in1mp/{g}.jpg", raw, JPG)
    bald_p = f"inq/refs/{g}__bald.jpg"
    if not os.path.exists(bald_p):
        b, _ = K.edit([raw], L.BALD_PROMPT, CALL1_SEED, canvas="v33")
        cv2.imwrite(bald_p, cv2.resize(b, (raw.shape[1], raw.shape[0]),
                                       interpolation=cv2.INTER_AREA), JPG)
    bald = cv2.imread(bald_p)
    d = np.abs(raw.astype(np.float32) - bald.astype(np.float32))
    prod, cranium = crop_bc(bald, f"inq_prod_{g}")
    nobald = crop_nobald(raw, f"inq_nobald_{g}")
    cv2.imwrite(f"inq/refs/{g}__PROD.jpg", prod, JPG)
    cv2.imwrite(f"inq/refs/{g}__NOBALD.jpg", nobald, JPG)
    inq[g] = {"cranium_used": bool(cranium),
              "bald_mad": round(float(d.mean()), 2),
              "bald_changed_pct": round(float((d.max(axis=2) > 16).mean()) * 100, 1),
              "prod_wh": [prod.shape[1], prod.shape[0]],
              "nobald_wh": [nobald.shape[1], nobald.shape[0]],
              "prod_white": round(white_frac(prod), 3),
              "nobald_white": round(white_frac(nobald), 3)}
    print(f"{g}  cranium={inq[g]['cranium_used']}  bald MAD {inq[g]['bald_mad']:6.2f}  "
          f"changed {inq[g]['bald_changed_pct']:5.1f}%  "
          f"PROD {prod.shape[1]}x{prod.shape[0]} white {inq[g]['prod_white']:.2f}  "
          f"NOBALD {nobald.shape[1]}x{nobald.shape[0]} white {inq[g]['nobald_white']:.2f}")
json.dump(inq, open("inq/meta/garment_only.json", "w"), indent=1)

In [ ]:
for p in PEOPLE:
    src = f"run/inputs/{p}.jpg"
    dst = f"inq/in1mp/{p}.jpg"
    if not os.path.exists(dst):
        cv2.imwrite(dst, V.normalise(cv2.imread(src)), JPG)

n = 0
t0 = time.time()
for g in PRODUCT:
    for p in PEOPLE:
        person = cv2.imread(f"inq/in1mp/{p}.jpg")
        for seed in SEEDS:
            for arm in ("PROD", "NOBALD"):
                out = f"inq/gen/{p}+{g}__{arm}__s{seed}.jpg"
                if os.path.exists(out):
                    continue
                ref = cv2.imread(f"inq/refs/{g}__{arm}.jpg")
                im, _ = K.edit([person, ref], V36.ER, seed, canvas="fal")
                cv2.imwrite(out, im, JPG)
                n += 1
print(f"{n} edits in {(time.time() - t0) / 60:.1f} min")
fired = [g for g, v in inq.items() if v["cranium_used"]]
touched = [g for g, v in inq.items() if v["bald_changed_pct"] > 5]
print(f"\nVERDICT  the parser found a head on {len(fired)}/{len(PRODUCT)} product shots {fired or ''}")
print(f"         the bald pass changed more than 5% of pixels on {len(touched)}/{len(PRODUCT)} {touched or ''}")
print(f"         whether that harms the try-on is the marking page's question (PROD vs NOBALD)")

## 9 · Verdicts

In [ ]:
print(f"{'1  onnxruntime-gpu':44s} {chosen}")
print(f"{'1  BiRefNet / SCHP provider':44s} {biref_prov} / {parser_prov}")
print(f"{'1  crops GPU vs CPU':44s} {gpu_biref + gpu_parser:.2f}s vs {cpu_biref + cpu_parser:.2f}s")
print(f"{'1  crop parity with the record (T1)':44s} {'PASS' if gpu_parity else 'FAIL'} "
      f"({len(parity) - len(failed)}/{len(parity)} within MAD 4.0)")
print(f"{'2  upscale vs none':44s} {made['NOSCALE']} pairs to mark ({noop} no-ops)")
print(f"{'3  one crop vs two':44s} {made['CROP2']} pairs to mark")
print(f"{'4  product shots, parser found a head':44s} {len(fired)}/{len(PRODUCT)}")
print(f"{'4  product shots, bald pass altered >5%':44s} {len(touched)}/{len(PRODUCT)}")
print(f"{'4  pairs to mark (PROD vs NOBALD)':44s} {len(glob.glob('inq/gen/*.jpg'))}")
print(f"\n{'total klein calls':44s} {v39_cost['klein_calls'] + len(PRODUCT) + n}")
print(f"{'wall':44s} {(time.time() - t0) / 60 + v39_cost['wall_minutes']:.0f} min")
print("\nInquiries 2, 3 and 4 are answered by marking, not by this run: build the page next.")

## 10 · Zip to Drive

In [ ]:
TERMINATE_WHEN_DONE = True  #@param {type:"boolean"}
DOWNLOAD_GRACE_SECONDS = 120  #@param {type:"integer"}

name = f"inquiry_{time.strftime('%Y%m%d_%H%M')}"
zip_path = f"/content/{name}.zip"
KEEP = ("__1crop.", "__2crop.", "__bald_1crop.", "__bald_2crop.")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir("run/gen"):
        z.write("run/gen/" + f, "v39/gen/" + f)
    for f in os.listdir("run/refs"):
        if any(k in f for k in KEEP):
            z.write("run/refs/" + f, "v39/refs/" + f)
    for f in os.listdir("run/in1mp"):
        z.write("run/in1mp/" + f, "v39/in1mp/" + f)
    for f in os.listdir("run/meta"):
        z.write("run/meta/" + f, "v39/meta/" + f)
    z.write(MATRIX, "v39/" + MATRIX)
    for sub in ("gen", "refs", "in1mp", "inputs", "meta"):
        for f in os.listdir(f"inq/{sub}"):
            z.write(f"inq/{sub}/{f}", f"inquiry/{sub}/{f}")
with zipfile.ZipFile(zip_path) as z:
    if z.testzip() is not None:
        raise RuntimeError("the zip is corrupt")
    names = z.namelist()
    for need in ("v39/meta/v39_meta.json", "v39/meta/cost_v39.json", "inquiry/meta/garment_only.json"):
        if need not in names:
            raise RuntimeError(f"missing {need}")
size_mb = os.path.getsize(zip_path) / 1e6
drive_copy = None
try:
    os.makedirs(os.path.join(BASE, "v3_runs"), exist_ok=True)
    drive_copy = os.path.join(BASE, "v3_runs", f"{name}.zip")
    shutil.copy(zip_path, drive_copy)
except Exception as e:
    print(f"Drive copy failed ({e}); the browser download is the only copy")
print(f"{name}.zip · {len(names)} files · {size_mb:.1f} MB")
if drive_copy:
    print(f"Drive  {drive_copy}")
print(f"Local  {zip_path}")

from google.colab import files
downloaded = False
try:
    files.download(zip_path)
    downloaded = True
except Exception as e:
    print(f"download failed ({e}) — take it from Drive or /content")

if downloaded:
    print(f"downloading; waiting {DOWNLOAD_GRACE_SECONDS}s before the runtime is released")
    time.sleep(DOWNLOAD_GRACE_SECONDS)

if TERMINATE_WHEN_DONE and (downloaded or drive_copy):
    from google.colab import runtime
    print("releasing the GPU")
    runtime.unassign()
else:
    print("runtime kept — nothing was saved off this machine")
